# Mini Project 7: Blood Cell Detection with YOLO26 (Option A - BCCD Dataset)
**Group 2: Aristide Kanamugire & Nicky Cheng**  
**Course:** COMP 9130 — Applied Artificial Intelligence  
**Dataset:** complete blood cell count.v1i.yolov8 (from Google Drive – Roboflow BCCD format, 3 classes: RBC, WBC, Platelets)


**1. Mount Google Drive & Setup**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


!pip install -U ultralytics --quiet

# Import after install
from ultralytics import YOLO

import os
from IPython.display import Image, display
import yaml
import glob
from collections import defaultdict

print(" Ultralytics imported successfully!")
print("Version info:", YOLO.__module__)


try:
    test_model = YOLO('yolo26n.pt')
    print("Test load successful (yolo26n.pt)")
except Exception as e:
    print("Import/load test failed:", e)

Mounted at /content/drive
 Ultralytics imported successfully!
Version info: ultralytics.models.yolo.model
Test load successful (yolo26n.pt)


**2. Dataset Preparation**

In [ ]:

dataset_folder_name = "complete blood cell count.v1i.yolov8"
base_drive_path = "/content/drive/MyDrive"
dataset_path = os.path.join(base_drive_path, dataset_folder_name)

print(f"Expected dataset location: {dataset_path}")


clean_name = "bccd_dataset"
clean_path = os.path.join(base_drive_path, clean_name)

if os.path.exists(dataset_path) and not os.path.exists(clean_path):
    os.rename(dataset_path, clean_path)
    dataset_path = clean_path
    print(f"Renamed folder to: {dataset_path}")
elif os.path.exists(clean_path):
    dataset_path = clean_path
    print("Using already renamed clean path")
else:
    print("Warning: folder not found – double-check Drive path and name")

# Verify basic structure
print("\nFolder contents:")
!ls -l "{dataset_path}"

data_yaml_path = os.path.join(dataset_path, "data.yaml")
if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f"data.yaml missing! Check folder: {dataset_path}")

print("\nLoading data.yaml...")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset config:")
print(data_config)

# Get class names & order dynamically
class_names = data_config.get('names', [])
nc = data_config.get('nc', len(class_names))
print(f"\nClasses ({nc}): {class_names}")

Expected dataset location: /content/drive/MyDrive/complete blood cell count.v1i.yolov8
Using already renamed clean path

Folder contents:
total 14
-rw------- 1 root root  330 Mar  4 01:34 data.yaml
-rw------- 1 root root  184 Mar  4 01:34 README.dataset.txt
-rw------- 1 root root 1004 Mar  4 01:34 README.roboflow.txt
drwx------ 2 root root 4096 Mar  4 01:34 test
drwx------ 2 root root 4096 Mar  4 01:49 train
drwx------ 2 root root 4096 Mar  4 01:50 valid

Loading data.yaml...
Dataset config:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['Platelets', 'RBC', 'WBC'], 'roboflow': {'workspace': 'university-of-macau-4pryf', 'project': 'complete-blood-cell-count', 'version': 1, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/university-of-macau-4pryf/complete-blood-cell-count/dataset/1'}}

Classes (3): ['Platelets', 'RBC', 'WBC']


**3. Model Training (YOLO26s from COCO pretrained)**
Minimum 25 epochs.

In [ ]:
def count_labels(split):
    counts = defaultdict(int)
    label_dir = os.path.join(dataset_path, split, "labels")
    if not os.path.exists(label_dir):
        return counts
    for txt_file in os.listdir(label_dir):
        if not txt_file.endswith('.txt'):
            continue
        with open(os.path.join(label_dir, txt_file), 'r') as f:
            for line in f:
                if line.strip():
                    cls_id = int(line.split()[0])
                    if 0 <= cls_id < nc:
                        counts[class_names[cls_id]] += 1
    return dict(counts)

print("Class distribution:")
for split in ['train', 'valid', 'test']:
    cnt = count_labels(split)
    total = sum(cnt.values())
    print(f"{split.capitalize()}: {cnt}  (total objects: {total})")

Class distribution:
Train: {'WBC': 244, 'Platelets': 255, 'RBC': 2640}  (total objects: 3139)
Valid: {'Platelets': 49, 'WBC': 63, 'RBC': 700}  (total objects: 812)
Test: {'RBC': 791, 'WBC': 61, 'Platelets': 55}  (total objects: 907)


**4. Evaluation  (mAP, Precision, Recall, Per-class, Confusion Matrix)**

In [ ]:
model = YOLO('yolo26s.pt')

results = model.train(
    data=data_yaml_path,
    epochs=25,
    imgsz=320,
    batch=4,
    name='bccd_yolo26s_group2_320',
    pretrained=True,
    optimizer='auto',
    seed=42,
    verbose=True,
    project='/content/runs/detect'
)

print(" Training completed!")
print("Best weights:", model.best)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/bccd_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bccd_yolo26s_group2_320, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

AttributeError: 'DetectionModel' object has no attribute 'best'

**Detailed Individual Training Curves**

In [ ]:
from ultralytics.utils.plotting import plot_results

run_dir = "/content/runs/detect/bccd_yolo26s_group2_320"
csv_file = os.path.join(run_dir, "results.csv")

if os.path.exists(csv_file):
    print("Generating EXACT same plots as your example report...")
    plot_results(file=csv_file)


    save_folder = os.path.join(run_dir, "individual_plots_for_report")
    os.makedirs(save_folder, exist_ok=True)

    print(f"\nSaving high-quality plots to:\n{save_folder}\n")

    for i, fig_num in enumerate(plt.get_fignums()):
        fig = plt.figure(fig_num)
        title = fig.canvas.get_window_title() or f"plot_{i+1}"
        filename = f"{title.replace(' ', '_').replace('/', '_')}.png"
        save_path = os.path.join(save_folder, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {filename}")

    print("\n All plots saved! Download this folder for your Google Doc.")
else:
    print("Run training first.")

**5.Evaluation – Metrics**

In [ ]:
val_results = model.val()

print("Overall Metrics:")
print(f"mAP@50     : {val_results.box.map50:.4f}")
print(f"mAP@50-95  : {val_results.box.map:.4f}")
print(f"Precision  : {val_results.box.mp:.4f}")
print(f"Recall     : {val_results.box.mr:.4f}")

print("\nPer-class mAP@50:")
for i, name in enumerate(data_config['names']):
    print(f"  {name}: {val_results.box.maps[i]:.4f}")

## 6. Training Curves

In [ ]:
big_curve = os.path.join(run_dir, "results.png")
if os.path.exists(big_curve):
    display(Image(filename=big_curve, width=900))

**7. Prediction Examples (at least 6 on validation images)**

In [ ]:
val_images = glob.glob(os.path.join(dataset_path, "valid", "images", "*.jpg"))[:8]

for img_path in val_images:
    preds = model.predict(img_path, conf=0.5, save=True)
    saved_img = os.path.join(preds[0].save_dir, os.path.basename(img_path))
    if os.path.exists(saved_img):
        display(Image(filename=saved_img, width=500))

**8. Confusion Matrix & PR Curve**

In [ ]:
display(Image(filename=os.path.join(run_dir, "confusion_matrix.png"), width=700))
display(Image(filename=os.path.join(run_dir, "PR_curve.png"), width=900))

print("Notebook complete!")
print("Download the folder 'individual_plots_for_report' for your report.")